In [2]:
import os
import pandas as pd
import numpy as np

INPUT_PATH  = "../../../data/phase2/labeled_signals.parquet"
OUTPUT_PATH = "../../../data/phase2/features_for_model.parquet"

FEATURE_COLS = [
    "ema_ratio", "rsi_14", "macd_hist", "atr_14",
    "session_quality_enc", "direction_enc", "signal_valid_enc",
]
LABEL_COL = "label"
META_COLS = ["date", "s3_key", "fold", "split"]

In [3]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived and encoded columns to df. Returns a new DataFrame.
    Input df must contain: ema_20, ema_50, rsi_14, macd_hist, atr_14,
    session_quality, direction, signal_valid, label.
    """
    out = df.copy()
    out["ema_ratio"]           = out["ema_20"] / out["ema_50"]
    out["session_quality_enc"] = out["session_quality"].map({"high": 2, "medium": 1, "low": 0})
    out["direction_enc"]       = out["direction"].map({"buy": 1, "sell": -1, "none": 0})
    out["signal_valid_enc"]    = out["signal_valid"].astype(int)
    return out


def assign_walk_forward_folds(
    df: pd.DataFrame,
    train_months: int = 6,
    test_months: int = 1,
) -> pd.DataFrame:
    """
    Assign walk-forward fold metadata. Returns an exploded DataFrame where each
    row appears once per fold it participates in (as train or test).
    A row may appear as train in multiple folds. Each fold is fully independent.
    Rows that don't fall into any fold window are excluded.
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["_orig"] = range(len(df))

    min_date = df["date"].min().to_period("M")
    max_date = df["date"].max().to_period("M")
    month_period = df["date"].dt.to_period("M")

    fold_idx, cursor, fold_windows = 0, min_date, []
    while True:
        train_end = cursor + train_months
        test_end  = train_end + test_months
        if test_end > max_date + 1:
            break
        fold_windows.append((fold_idx, cursor, train_end, train_end, test_end))
        fold_idx += 1
        cursor += test_months

    records = []
    for fi, train_start, train_end, test_start, test_end in fold_windows:
        train_idx = df.index[(month_period >= train_start) & (month_period < train_end)]
        test_idx  = df.index[(month_period >= test_start)  & (month_period < test_end)]
        for i in train_idx:
            records.append({"_orig": i, "fold": fi, "split": "train"})
        for i in test_idx:
            records.append({"_orig": i, "fold": fi, "split": "test"})

    fold_df = pd.DataFrame(records)
    result = df.merge(fold_df, on="_orig", how="inner")
    return result.drop(columns=["_orig"]).reset_index(drop=True)


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full feature prep pipeline:
    1. Drop rows with NaN label (direction=="none")
    2. Encode features
    3. Assign walk-forward folds
    4. Return DataFrame with META_COLS + FEATURE_COLS + LABEL_COL

    Does NOT scale features — scaling happens inside each fold during training
    to prevent leakage.
    """
    df = df[df["label"].notna()].copy()
    df = encode_features(df)
    df = assign_walk_forward_folds(df)
    keep = META_COLS + FEATURE_COLS + [LABEL_COL]
    return df[keep].reset_index(drop=True)

In [4]:
raw = pd.read_parquet(INPUT_PATH)
prepped = prepare_features(raw)

print(f"Total rows after dropping direction=none: {len(prepped)}")
print(f"\nFold distribution:")
print(prepped.groupby(["fold", "split"]).size().to_string())
print(f"\nLabel distribution:\n{prepped['label'].value_counts()}")

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
prepped.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

Total rows after dropping direction=none: 37235

Fold distribution:
fold  split
0     test      71
      train    444
1     test      67
      train    462
2     test      84
      train    457
3     test      52
      train    487
4     test      65
      train    439
5     test      78
      train    421
6     test      81
      train    417
7     test      67
      train    427
8     test      28
      train    427
9     test     102
      train    371
10    test     108
      train    421
11    test      79
      train    464
12    test      57
      train    465
13    test      94
      train    441
14    test      76
      train    468
15    test      66
      train    516
16    test      98
      train    480
17    test     106
      train    470
18    test      60
      train    497
19    test      63
      train    500
20    test      81
      train    469
21    test      85
      train    474
22    test      72
      train    493
23    test      61
      train    467
24    te

/var/folders/67/xsn6f8gs2t94xg1kmbx02mjw0000gn/T/ipykernel_91313/893240860.py:30: UserWarning: Converting to Period representation will drop timezone information.
  min_date = df["date"].min().to_period("M")
/var/folders/67/xsn6f8gs2t94xg1kmbx02mjw0000gn/T/ipykernel_91313/893240860.py:31: UserWarning: Converting to Period representation will drop timezone information.
  max_date = df["date"].max().to_period("M")
/var/folders/67/xsn6f8gs2t94xg1kmbx02mjw0000gn/T/ipykernel_91313/893240860.py:32: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  month_period = df["date"].dt.to_period("M")


In [5]:
df = pd.read_parquet(OUTPUT_PATH)
print(df[META_COLS + FEATURE_COLS + [LABEL_COL]].head(10).to_string())
print(f"\nFeature dtypes:\n{df[FEATURE_COLS].dtypes}")
print(f"\nAny NaN in features: {df[FEATURE_COLS].isna().any().any()}")

                       date  s3_key  fold  split  ema_ratio     rsi_14  macd_hist    atr_14  session_quality_enc  direction_enc  signal_valid_enc  label
0 2020-01-31 00:00:00+00:00  EURUSD     0  train   0.998687  37.557690  -0.001199  0.005070                    0             -1                 0    1.0
1 2020-02-04 00:00:00+00:00  EURUSD     0  train   0.998744  45.787164  -0.000359  0.005169                    0             -1                 0    1.0
2 2020-02-05 00:00:00+00:00  EURUSD     0  train   0.998631  43.389042  -0.000269  0.005261                    0             -1                 0    0.0
3 2020-02-06 00:00:00+00:00  EURUSD     0  train   0.998311  37.335251  -0.000468  0.005469                    0             -1                 0    0.0
4 2020-02-07 00:00:00+00:00  EURUSD     0  train   0.997941  35.069773  -0.000674  0.005474                    0             -1                 0    0.0
5 2020-02-07 00:00:00+00:00  EURUSD     1  train   0.997941  35.069773  -0.000674 